In [1]:
import gdsfactory as gf
import kfactory as kf
from kfactory.utils.fill import fill_tiled
from helpers import wafer_spec 
from helpers.die_estimator import estimate_max_dies_on_wafer, plot_die_packing
from flexgrid import flexgrid
from EKST_v2_BRT import ekst_v2_brt_master
from EKST_v2_PUL import ekst_v2_pul_master
from ekin_master_die import edge_coupler_array_ekn_def_butt, edge_coupler_array_ekn_def_butt_3loops, edge_coupler_array_ekn_def_centerskip
from logo_maker import svg_logo
from wafer_component import wafer_from_spec


c:\Users\BosdurmazEB\AppData\Local\Programs\Python\Python311\Lib\site-packages\kfactory\decorators.py:406: UserWarning: `width` overrides `start_width`. Use only `start_width` going forward.
  cell = f(**params)  # type: ignore[call-arg]


In [2]:
WAFER_ID = "ASHW_v0_W00"
widths = (9, 11, 12, 13)
SIN_THICKNESS = 80
TEOS_THICKNESS = 3000
PECVD_OXIDE_THICKNESS = 5000

In [3]:
my_die = ekst_v2_pul_master()
wafer = wafer_spec.make_semi_wafer_spec("100mm", edge_exclusion_um=3000, use_secondary_flat=False)
result = estimate_max_dies_on_wafer(
    die=my_die,
    wafer=wafer,
    allow_rotation=True,
    offset_samples=11,
)

wafer_filled= wafer_from_spec(wafer=wafer)
wafer_filled.locked = False
assign_array = result.index_map.copy()

d:\CleanroomFabrication\mesapdk-lab\spirals.py:43: UserWarning: {'width': 0.75} ignored for cross_section 'xs_ekn300_te_IMGREV'
  xs = gf.get_cross_section(cross_section, width = width)
c:\Users\BosdurmazEB\AppData\Local\Programs\Python\Python311\Lib\site-packages\gdsfactory\components\bends\bend_euler.py:110: UserWarning: {'width': 0.75} ignored for cross_section 'xs_ekn300_te_IMGREV'
  x = gf.get_cross_section(cross_section, width=width or x.width)


0.75
0.75
0.75
0.75
0.75
0.75
0.75
0.75
0.75
0.75


c:\Users\BosdurmazEB\AppData\Local\Programs\Python\Python311\Lib\site-packages\gdsfactory\components\waveguides\straight.py:33: UserWarning: {'width': 0.75} ignored for cross_section 'xs_ekn300_te_IMGREV'
  x = gf.get_cross_section(cross_section, width=width)
c:\Users\BosdurmazEB\AppData\Local\Programs\Python\Python311\Lib\site-packages\gdsfactory\components\bends\bend_euler.py:110: UserWarning: {'width': 2.5} ignored for cross_section 'xs_c5bc0727'
  x = gf.get_cross_section(cross_section, width=width or x.width)


In [4]:
logo = svg_logo(
        svg_path="./static/AQO_logo2.svg",
        layer=(5,0),
        target_width_um=1500.0,   # final width in um
        resolution=0.08,         # smaller -> smoother curves
        center=True,
    )

In [5]:
BRT_TAP = gf.partial(ekst_v2_brt_master,ext_grp_spacing=127, 
                     label = f"{WAFER_ID.split('_')[0] + '_' + WAFER_ID.split('_')[1]}\nBRT TAP",
                     chip_id_label = WAFER_ID.split('_')[-1],
                     logo=logo, 
                     bend_rads=(2000,2000,2000,2000,2000,2000,2000),
                     widths=widths,
                     logo_loc=(8500,-3750))

BRT_BUT = gf.partial(ekst_v2_brt_master,
                     ext_grp_spacing=127,
                     ec_array_def=edge_coupler_array_ekn_def_butt, 
                     label = f"{WAFER_ID.split('_')[0] + '_' + WAFER_ID.split('_')[1]}\nBRT BUT",
                     chip_id_label = WAFER_ID.split('_')[-1],
                     logo=logo, 
                     bend_rads=(2000,2000,2000,2000,2000,2000,2000),
                     widths=widths,
                     logo_loc=(8500,-3750))



In [7]:
assign_array[(0, -3)] = BRT_BUT
assign_array[(1, -3)] = BRT_BUT
assign_array[(0, -2)] = BRT_BUT
assign_array[(1, -2)] = BRT_BUT
assign_array[(-1, -1)] = BRT_BUT
assign_array[(0, -1)] =BRT_BUT
assign_array[(1, -1)] = BRT_BUT
assign_array[(2, -1)] = BRT_BUT
assign_array[(-1, 0)] = BRT_BUT
assign_array[(0, 0)] = BRT_BUT
assign_array[(1, 0)] = BRT_TAP
assign_array[(2, 0)] = BRT_TAP
assign_array[(-1, 1)] = BRT_TAP
assign_array[(0, 1)] = BRT_TAP
assign_array[(1, 1)] = BRT_TAP
assign_array[(2, 1)] = BRT_TAP
assign_array[(-1, 2)] = BRT_TAP
assign_array[(0, 2)] = BRT_TAP
assign_array[(1, 2)] = BRT_TAP
assign_array[(2, 2)] = BRT_TAP
assign_array[(0, 3)] = BRT_TAP
assign_array[(1, 3)] = BRT_TAP
assign_array[(0, 4)] = BRT_TAP
assign_array[(1, 4)] = BRT_TAP

In [8]:
for die in assign_array:
    if assign_array[die] !=None:
        die_name = ("{}_I{}_{}\nX{:.1f} Y{:.1f}".format(WAFER_ID.split('_')[-1], die[0], die[1],
                                                         float(result.get_center(die)[0])/1000, 
                                                         float(result.get_center(die)[1])/1000))
        dieref = wafer_filled.add_ref(assign_array[die](chip_id_label = die_name,))
        dieref.dmove(origin=(0,0), destination=result.get_center(die))

c:\Users\BosdurmazEB\AppData\Local\Programs\Python\Python311\Lib\site-packages\kfactory\decorators.py:406: UserWarning: `width` overrides `start_width`. Use only `start_width` going forward.
  cell = f(**params)  # type: ignore[call-arg]


In [9]:
"""
Adding wafer-level labels and logo.
Currently done manually, in future might be automated.
"""
wlabel_text = gf.partial(gf.components.text, layer="LABEL_SIN")
wafer_label = wlabel_text(size=1000, 
                          text = WAFER_ID, 
                          justify = 'center' )

t_label = wlabel_text(size=750, 
                      text = 'Si3N4: {:.0f}nm'.format(SIN_THICKNESS), 
                      justify = 'left' )

teos_label = wlabel_text(size=750, 
                         text = ' TEOS: {:.0f}nm'.format(TEOS_THICKNESS), 
                         justify = 'left' )

pecvd_label = wlabel_text(size=750, 
                          text = 'PECVD_OX: {:.0f}nm'.format(PECVD_OXIDE_THICKNESS),
                          justify = 'left' )


label_block = flexgrid(components=(
                                        
                                        pecvd_label,
                                        teos_label,
                                        t_label,
                                        wafer_label
    ),
                                    spacing=(100,200),
                                    shape=(5,1),
                                    align_x='xmin',
                                    align_y='center'
                                    )

lb_ref = wafer_filled.add_ref(label_block, columns=2, rows=2, column_pitch=55000, row_pitch=-55000).dmove(origin=(0,0), destination=(-55000/2 - label_block.dxsize/2, 55000/2 - label_block.dysize/2))
lb_box_ref = wafer_filled.add_ref(gf.components.rectangle(size=(label_block.dxsize, label_block.dysize), layer='FLOORPLAN'), columns=2, rows=2, column_pitch=55000 , row_pitch=55000).dmove(origin=(0,0), destination=(lb_ref.dxmin, lb_ref.dymin))

""" 
GET a list of cells, which suppose to be processed by PEC in a loop 
and printing it to a file for later use in BEAMER
"""
dies_to_PEC = []

for inst in wafer_filled.insts:
    if "ekst_v2" in inst.cell_name:
        dies_to_PEC.append("True\t {}\n".format(inst.cell_name))
    else:
        dies_to_PEC.append("False\t {}\n".format(inst.cell_name))

In [12]:
wafer_filled.show()